### **Student Information**
GitHub ID: MCTsai412

Kaggle name: tsaimincheng

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Final result:** public leaderboard macro-F1 **0.6496**, 5-fold out-of-fold (OOF) macro-F1 **0.6420**, up from 0.5045 for the TF-IDF baseline. The private-leaderboard rank is in the snapshot at the top (added after the competition closes). The code below is the same code as the Kaggle notebook `tsaimincheng/dm-lab2-final`; a full re-run on Kaggle reproduced the submitted predictions exactly.

**Task:** classify 3,360 train / 1,440 test short Reddit posts into 6 emotions (anger, disgust, fear, joy, sadness, surprise). Metric: macro-F1.

---

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

- **Join:** `final_posts.json` (text), `data_identification.csv` (train/test) and `emotion.csv` (label) are merged on `post_id`.
- **No text cleaning.** Letter case (no lowercasing), punctuation, emojis and stop words are kept as they are, because every model uses a pretrained subword tokenizer trained on raw web text, and cues like "!!!" or ALL CAPS carry emotion.
- **Truncation:** 128 tokens for the fine-tuned models (posts are short: median 13 words, so under 1% are truncated), 256 for the frozen embedding model.
- **Class imbalance:** `joy` is about 3x as common as `fear`, so all models use balanced class weights.
- **Validation:** one stratified 5-fold split shared by all models, so every post gets exactly one OOF prediction and models and blends are compared on the same 3,360 rows. Early on I picked the best epoch on a single split, which overstated the score (0.62 validation vs 0.56 leaderboard), so every later model uses a fixed number of epochs.

### 1.2 Feature Engineering Steps

- **Instruction-prefixed LLM embeddings** (frozen model). Each post is embedded with `Qwen/Qwen3-Embedding-4B` as `Instruct: Classify the emotion expressed in this post (anger, ..., surprise)\nQuery: <post>`. The instruction steers the vector toward emotion instead of topic. The result is a 2,560-dimensional L2-normalised vector per post, computed once in fp16 and reused in every fold.
- **Subword tokens** (fine-tuned models): the model's own BPE tokens, tokenized per batch with dynamic padding.
- **Class weights:** `n / (n_classes x class_count)`, computed on each fold's training part, used in the loss and in logistic regression.

### 1.3 Explanation of Your Model

The final prediction is a fixed **50/50 average of class probabilities** from two different model types. Every model uses the same 5-fold CV, and a test prediction is the mean of its 5 fold models.

1. **Frozen Qwen3-Embedding-4B + logistic regression** (`C=4`, `class_weight="balanced"`). Small `C` works best because there are 2,560 features and only about 2,700 training rows (C from 0.5 to 4 gives 0.594-0.598, C=64 gives 0.583). OOF macro-F1 0.597.
2. **Three fine-tuned emotion-pretrained transformers:** `j-hartmann/emotion-english-distilroberta-base` (lr 3e-5, batch 32) and `j-hartmann/emotion-english-roberta-large` (lr 1e-5, batch 16, two seeds). These checkpoints already have an emotion head over {anger, disgust, fear, joy, neutral, sadness, surprise}, so I keep it, select our 6 columns by label name and drop `neutral`. Training: AdamW (weight decay 0.01), 10% warmup then linear decay, gradient clipping 1.0, fp16, class-weighted cross-entropy, a fixed 4 epochs. OOF macro-F1 0.615 / 0.627 / 0.627, mean of the three 0.633.

The two types are built very differently, so they make different mistakes and averaging cancels some of them. The blend gives OOF **0.642**. Blend weights were set by a rule (equal per type), not fitted on OOF; any emotion-model weight from 0.4 to 0.7 is within 0.002.

**How the score improved**

| Step | What changed | OOF macro-F1 | Public LB |
|---|---|---|---|
| 1 | TF-IDF + logistic regression baseline | - | 0.5045 |
| 2 | Fine-tuned `roberta-base`, single split | 0.6186 (best epoch, optimistic) | 0.5646 |
| 3 | 8 fine-tuned generic encoders (roberta / deberta-v3 / electra / bert), 5-fold | 0.6105 | 0.5977 |
| 4 | + frozen Qwen3-Embedding-4B + LR (0.7 / 0.3) | 0.6178 | 0.6173 |
| 5 | + 3 emotion-pretrained models (12 models) | 0.6395 | 0.6467 |
| **6** | **Ablation: keep only Qwen3-4B + 3 emotion models (final)** | **0.6420** | **0.6496** |

In a paired-bootstrap ablation (500 resamples) removing the 3 emotion models cost -0.022 (95% CI [-0.032, -0.012]), while removing the 8 generic encoders changed nothing (+0.0025, CI [-0.005, +0.010]). That is why the final blend has 4 models instead of 12. The leaderboard difference between the two (0.6496 vs 0.6467) is within noise, so I read it as "at least as good".

**What did not help:** deberta-v3-large (OOF 0.592, no better than base models); other `C` values or a second prompt for the same embedding (blends 0.615-0.619); stacking with logistic regression (best 0.613, below simple averaging); per-class probability scaling (nested-CV gain +0.005 +/- 0.012, unstable); QLoRA fine-tuning of `Qwen3-8B-Base` (fold 1 only: 0.588 vs 0.603-0.628 for the final models, 30 min per fold, so I stopped).

**Important caveat: overlap with GoEmotions.** 49.4% of the training posts and 51.5% of the test posts appear word for word in the public GoEmotions dataset (ids changed), and the `j-hartmann` models were trained on data that includes GoEmotions, so they have probably seen many of these posts. Splitting the OOF score by overlap:

| OOF macro-F1 | posts in GoEmotions | posts not in GoEmotions |
|---|---|---|
| Qwen3-Embedding-4B + LR | 0.639 | 0.545 |
| deberta-v3-large (generic) | 0.634 | 0.535 |
| j-hartmann emotion models (mean of 3) | 0.719 | 0.533 |
| final blend | 0.718 | 0.551 |

The emotion models gain far more on the overlapping half and are not better than generic models on the other half, so **most of the jump from 0.617 to 0.650 probably comes from these models having seen the posts, not from better emotion understanding.** To check this I built a clean version: GoEmotions mapped to the 6 Ekman classes, with every comment whose text appears in the competition data removed (31,639 left), used to pre-train `roberta-large` and `deberta-v3-large` before the usual 5-fold fine-tuning. That blend (Qwen + clean GoEmotions models) scores OOF 0.623, about 0.02 below the final blend, and that gap is roughly the advantage from having seen the overlapping posts. On posts not in GoEmotions, every model and blend I tried scores only 0.53-0.55. I kept the `j-hartmann` blend because the competition places no restriction on pretrained models, and I report the effect openly. I did not look up any test labels.

**Error analysis:** the largest confusion is anger vs disgust: about a quarter of disgust posts (107 of 454) are predicted as anger, and disgust has the lowest per-class F1 (0.49 vs 0.63-0.78 for the others). Many short posts express both, and labels are noisy.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I worked with Claude Code throughout. My prompts were in Chinese; they are paraphrased here. The AI ran the experiments on Kaggle and I decided what to keep.

**Experiments and discussion**
- I asked which model and validation score the 0.5646 run had. The AI asked for the validation macro-F1 and suggested logging every run. The log showed my validation score (0.6186) was far above the leaderboard (0.5646).
- I asked what to do after blending. The AI suggested more diverse models and 5-fold CV with out-of-fold (OOF) predictions to choose blends, and I used this from then on, with fixed epochs. I proposed pushing every run as a version of one Kaggle notebook, and the AI agreed.
- I felt the leaderboard was far ahead of me and that language models should not need fine-tuning. The AI checked the leaderboard (I was 8th of 10, the top 6 within 0.614-0.635) and read it as label noise limiting the score, not a wrong model family. It listed three directions: frozen embeddings, zero-shot LLMs, and emotion-pretrained checkpoints. It warned that the checkpoints carry rule risk and may share training data with the competition.
- I asked it to run the frozen-embedding test first, and whether other models or better preprocessing and features could help. Generic embeddings (e5, bge, mpnet) scored 0.48-0.57 and did not improve the blend. The AI saw little room in preprocessing (keep case, no URLs) and suggested larger LLM-based embeddings, so we tried `Qwen3-Embedding`. The leaderboard rose from 0.5977 to 0.6173, so I kept this direction.
- After checking that the rules do not forbid pretrained models, we tried the emotion-pretrained `j-hartmann` checkpoints. They gave the biggest gain, and ablations then showed the 8 generic models added nothing. Checking the AI's earlier overlap warning found that about half of the posts also appear in GoEmotions (see the caveat above).
- We tried deberta-v3-large, stacking, class scaling, extra prompt variants and our own GoEmotions pre-training. We agreed to run only one fold of QLoRA on Qwen3-8B and stop if it did not beat the other models; it did not (0.588).

**Final decision**
- **Kept:** Qwen3-Embedding-4B + logistic regression, and three fine-tuned emotion models, averaged 50/50 (OOF 0.6420, leaderboard 0.6496).
- **Rejected:** the generic fine-tuned models in the blend (no gain), deberta-v3-large, stacking, class scaling, extra prompts, own GoEmotions pre-training and QLoRA, because none beat the final blend in OOF.

The instruction text used for the embedding model is itself a prompt: `Instruct: Classify the emotion expressed in this post (anger, disgust, fear, joy, sadness, surprise)`. A second wording gave no gain on `Qwen3-Embedding-0.6B`, so I kept this one.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
# Final method: 0.5 x (Qwen3-Embedding-4B + logistic regression) + 0.5 x (3 fine-tuned emotion models).
# This is the code that produced the final submission (Kaggle notebook tsaimincheng/dm-lab2-final).
# It needs a GPU (16 GB T4 is enough); outputs are not stored here, see the numbers in the report.
import gc, glob, json, os, random, time
import numpy as np, pandas as pd, torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SEED, FOLDS, MAX_LEN = 42, 5, 128
dev = "cuda" if torch.cuda.is_available() else "cpu"
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(SEED)

# Join the three competition files on post_id: text + train/test flag + label (train only)
hits = glob.glob("/kaggle/input/**/emotion.csv", recursive=True) + glob.glob("data/kaggle/emotion.csv")
assert hits, "emotion.csv not found: put the competition data in data/kaggle/"
DATA = os.path.dirname(hits[0])

posts = pd.DataFrame(json.load(open(f"{DATA}/final_posts.json", encoding="utf8")))
df = (posts.merge(pd.read_csv(f"{DATA}/data_identification.csv"), on="post_id")
           .merge(pd.read_csv(f"{DATA}/emotion.csv"), on="post_id", how="left"))
train = df[df.set == "train"].reset_index(drop=True)
test = df[df.set == "test"].reset_index(drop=True)
assert len(train) == 3360 and len(test) == 1440 and train.label.notna().all() and test.label.isna().all()

classes = sorted(train.label.unique())                 # fixed column order for every probability matrix
y = train.label.map({c: i for i, c in enumerate(classes)}).values
texts, test_texts = train.text.tolist(), test.text.tolist()   # raw text: no cleaning on purpose

# One stratified 5-fold split shared by every model, so all out-of-fold (OOF) predictions are comparable
folds = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(texts, y))
print(classes, [len(va) for _, va in folds])
print(train.label.value_counts().to_dict())

## 2. Feature Engineering Steps

In [ ]:
# 2a. Instruction-prefixed LLM sentence embeddings (frozen Qwen3-Embedding-4B, 2560 features per post)
INSTR = "Instruct: Classify the emotion expressed in this post (anger, disgust, fear, joy, sadness, surprise)\nQuery: "

from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("Qwen/Qwen3-Embedding-4B", device=dev,
                          model_kwargs={"torch_dtype": torch.float16} if dev == "cuda" else {})
enc.max_seq_length = 256
embed = lambda tx: enc.encode([INSTR + t for t in tx], batch_size=16, normalize_embeddings=True,
                              show_progress_bar=False).astype(np.float32)
t0 = time.time()
X_qwen, Xt_qwen = embed(texts), embed(test_texts)      # computed once, reused in every fold (no labels involved)
print("embedding matrix:", X_qwen.shape, Xt_qwen.shape, f"{time.time() - t0:.0f}s")
del enc; gc.collect(); torch.cuda.empty_cache()

# 2b. Subword tokenization for the fine-tuned models: tokenize each batch on the fly, dynamic padding, truncate at MAX_LEN
def batches(tok, txt, ys=None, shuffle=False, bs=32):
    idx = np.random.permutation(len(txt)) if shuffle else np.arange(len(txt))
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        enc = tok([txt[j] for j in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)
        yield enc, (torch.tensor(ys[b]).to(dev) if ys is not None else None)

tok = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-roberta-large")
lens = np.array([len(tok(t)["input_ids"]) for t in texts])
print(f"tokens per post: median {np.median(lens):.0f}, 99th pct {np.percentile(lens, 99):.0f}, truncated by MAX_LEN: {(lens > MAX_LEN).mean():.2%}")

# 2c. Class weights (balanced, computed on each fold's training part) are used inside the models below
print("class weights (full train):", dict(zip(classes, compute_class_weight('balanced', classes=np.arange(6), y=y).round(2))))

## 3. Model Implementation Steps

In [ ]:
# 3a. Frozen Qwen3-Embedding-4B + class-balanced logistic regression (small C = strong regularisation, best in {0.5..64})
oof, tst = {}, {}

def cv_frozen(X, Xt, C):
    o, t = np.zeros((len(X), 6)), np.zeros((len(Xt), 6))
    for tr, va in folds:
        clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(X[tr], y[tr])
        o[va] = clf.predict_proba(X[va]); t += clf.predict_proba(Xt) / FOLDS   # test = mean of the 5 fold models
    return o, t

oof["qwen3_4b_lr"], tst["qwen3_4b_lr"] = cv_frozen(X_qwen, Xt_qwen, C=4)
print(f"Qwen3-Embedding-4B + LR   OOF macro-F1 {f1_score(y, oof['qwen3_4b_lr'].argmax(1), average='macro'):.4f}")

# 3b. Fine-tuned emotion-pretrained transformers (j-hartmann checkpoints).
# Keep their pretrained emotion head, select our 6 classes by label name and drop "neutral".
def run_fold(cfg, tok, tr_idx, va_idx):
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model"]).to(dev).float()
    lab = {v.lower(): k for k, v in model.config.id2label.items()}
    cols = [lab[c] for c in classes]
    logits = lambda enc: model(**enc).logits.float()[:, cols]

    w = torch.tensor(compute_class_weight("balanced", classes=np.arange(6), y=y[tr_idx]), dtype=torch.float).to(dev)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)                      # class-weighted loss for macro-F1
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    steps = cfg["epochs"] * ((len(tr_idx) + cfg["bs"] - 1) // cfg["bs"])
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)   # 10% warmup, then linear decay
    scaler = torch.amp.GradScaler(enabled=dev == "cuda")
    trt = [texts[i] for i in tr_idx]
    for ep in range(cfg["epochs"]):                                    # fixed epochs: no best-epoch picking on validation data
        model.train()
        for enc, yb in batches(tok, trt, y[tr_idx], shuffle=True, bs=cfg["bs"]):
            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
                loss = loss_fn(logits(enc), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()

    @torch.no_grad()
    def predict(txt):
        model.eval()
        return np.concatenate([torch.softmax(logits(enc), -1).cpu().numpy() for enc, _ in batches(tok, txt, bs=64)])
    out = predict([texts[i] for i in va_idx]), predict(test_texts)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return out

RUNS = {
    "emo_distilroberta_s42": dict(model="j-hartmann/emotion-english-distilroberta-base", seed=42, lr=3e-5, epochs=4, bs=32),
    "emo_roberta_large_s42": dict(model="j-hartmann/emotion-english-roberta-large", seed=42, lr=1e-5, epochs=4, bs=16),
    "emo_roberta_large_s1":  dict(model="j-hartmann/emotion-english-roberta-large", seed=1,  lr=1e-5, epochs=4, bs=16),
}
for name, cfg in RUNS.items():
    seed_all(cfg["seed"]); tok = AutoTokenizer.from_pretrained(cfg["model"])
    o, t = np.zeros((len(texts), 6)), np.zeros((len(test_texts), 6))
    for k, (tr, va) in enumerate(folds):
        t0 = time.time()
        o[va], pt = run_fold(cfg, tok, tr, va); t += pt / FOLDS
        print(f"{name} fold {k + 1}/{FOLDS}  F1 {f1_score(y[va], o[va].argmax(1), average='macro'):.4f}  {time.time() - t0:.0f}s", flush=True)
    oof[name], tst[name] = o, t
    print(f"== {name}  OOF macro-F1 {f1_score(y, o.argmax(1), average='macro'):.4f}", flush=True)

In [ ]:
# 3c. Blend: fixed 50/50 average of the two model types (weights set by a rule, not fitted on OOF), then write the submission
f1 = lambda p: f1_score(y, p.argmax(1), average="macro")
EMO = [k for k in oof if k.startswith("emo_")]
blend_oof = 0.5 * oof["qwen3_4b_lr"] + 0.5 * np.mean([oof[k] for k in EMO], 0)
blend_tst = 0.5 * tst["qwen3_4b_lr"] + 0.5 * np.mean([tst[k] for k in EMO], 0)

print(pd.Series({k: f1(v) for k, v in oof.items()} | {"emotion models (mean)": f1(np.mean([oof[k] for k in EMO], 0)),
                                                       "FINAL BLEND": f1(blend_oof)}, name="OOF macro-F1").round(4).to_string())
print(classification_report(y, blend_oof.argmax(1), target_names=classes, digits=3))
print(pd.DataFrame(confusion_matrix(y, blend_oof.argmax(1)), index=classes, columns=classes))

sub = pd.DataFrame({"id": test.post_id.values, "label": [classes[i] for i in blend_tst.argmax(1)]})
assert len(sub) == 1440 and sub.label.isin(classes).all()
sub.to_csv("submission.csv", index=False)
print(sub.label.value_counts().to_dict())